In [1]:
import numpy as np
from scipy.interpolate import griddata
import numpy as np
import torch
import sys, os
sys.path.append(os.path.abspath(".."))  # project root
print(os.path.abspath(".."))
from datasets.data import SpatialDataset
from datasets.transformerRegressorDataClass import TransformerPointDataset
from scipy.interpolate import Rbf
from scipy.interpolate import LinearNDInterpolator

/home/user_116/Project-B-Technion/Transformer_Map_Interp


In [2]:
def _to_numpy(x):
    # works for numpy, torch tensors, lists
    if hasattr(x, "detach"):  # torch.Tensor
        return x.detach().cpu().numpy()
    return np.asarray(x)

def _scalar(x):
    if x is None:
        return np.nan
    a = np.asarray(x)
    if a.size == 0:
        return np.nan
    # robustly take the first element (handles (), (1,), (1,1), etc.)
    return float(a.ravel()[0])


def idw_local_all(nei_coords_list, nei_y_list, query_coords, p=2.0, eps=1e-12):
    """
    Local IDW using *all* precomputed neighbors for each target point.
    """
    N = len(nei_coords_list)
    preds = np.zeros(N, dtype=float)
    qxy = _to_numpy(query_coords)

    for i in range(N):
        nei_xy = _to_numpy(nei_coords_list[i]).astype(float)
        nei_y  = _to_numpy(nei_y_list[i]).astype(float).reshape(-1)
        #print(nei_xy.size)
        if nei_xy.size == 0:
            preds[i] = np.nan
            continue

        d = np.linalg.norm(nei_xy - qxy[i], axis=1)  # (S,)

        # exact match protection
        if np.any(d < 1e-12):
            preds[i] = float(nei_y[d.argmin()])
            continue

        w = 1.0 / (d**p + eps)
        preds[i] = float(np.sum(w * nei_y) / np.sum(w))

    return preds

def linear_local(nei_coords_list, nei_y_list, query_coords, fill_with_idw=True, p=2.0):
    """
    Local linear interpolation using precomputed neighbors.
    - nei_coords_list: list of (S_i, 2)
    - nei_y_list: list of (S_i,)
    - query_coords: (N, 2)
    - fill_with_idw: if True, fill NaN/extrapolation with local IDW
    """
    N = len(nei_coords_list)
    preds = np.zeros(N, dtype=float)
    qxy = _to_numpy(query_coords)
    counter_NaN=0
    for i in range(N):
        nei_xy = _to_numpy(nei_coords_list[i]).astype(float)
        nei_y  = _to_numpy(nei_y_list[i]).astype(float).reshape(-1)

        if len(nei_xy) < 3:
            # not enough points to form a triangle; fallback to mean
            preds[i] = np.mean(nei_y)
            continue

        #try:
        interp = LinearNDInterpolator(nei_xy, nei_y, fill_value=np.nan)
        pred = interp(qxy[i])
        
        if np.isnan(pred) and fill_with_idw:
            #print("NaN")
            counter_NaN+=1
            print(f"Total NaN so far: {counter_NaN}")
            # fallback to local IDW if outside convex hull
            # d = np.linalg.norm(nei_xy - qxy[i], axis=1)
            # w = 1.0 / (d**p + 1e-12)
            # pred = np.sum(w * nei_y) / np.sum(w)
            d = np.linalg.norm(nei_xy - qxy[i], axis=1)
            w = 1.0 / (d**p + 1e-12)
            pred = np.sum(w * nei_y) / np.sum(w)
        # except Exception:
        #     # triangulation sometimes fails if neighbors are colinear
        #     d = np.linalg.norm(nei_xy - qxy[i], axis=1)
        #     w = 1.0 / (d**p + 1e-12)
        #     pred = np.sum(w * nei_y) / np.sum(w)

        preds[i] = _scalar(pred)
        print(f"Total NaN: {counter_NaN}")
    return preds

def rbf_local(nei_coords_list, nei_y_list, query_coords, function='linear'):
    preds = np.zeros(len(nei_coords_list))
    qxy = _to_numpy(query_coords)
    for i, (xy, y) in enumerate(zip(nei_coords_list, nei_y_list)):
        xy = _to_numpy(xy); y = _to_numpy(y).reshape(-1)
        if len(xy) < 3:
            preds[i] = np.mean(y)
            continue
        try:
            rbf = Rbf(xy[:,0], xy[:,1], y, function=function)
            preds[i] = float(rbf(qxy[i,0], qxy[i,1]))
        except Exception:
            preds[i] = np.mean(y)
    return preds

def MSE(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

def MAP(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred))/len(y_true)

In [4]:
# Show the current working directory
from pathlib import Path
print(os.getcwd())
os.chdir("..")
sys.path.append(os.path.abspath(".."))  # project root
print(os.path.abspath(".."))
p_cache = Path("cache")
print("cache exists?", p_cache.exists(), "->", p_cache.resolve())
# trainset = torch.load(r"./cache/trainset_n32_e035_1arc_v3_cropped_train_n32_e035_1arc_v3_cropped_val_n32_e035_1arc_v3_cropped_test_keep_n0.05_seed5.pt",map_location="cpu",weights_only=False)
# validset = torch.load(r"./cache/validset_n32_e035_1arc_v3_cropped_train_n32_e035_1arc_v3_cropped_val_n32_e035_1arc_v3_cropped_test_keep_n0.05_seed5.pt",map_location="cpu",weights_only=False)
# testset = torch.load(r"./cache/testset_n32_e035_1arc_v3_cropped_train_n32_e035_1arc_v3_cropped_val_n32_e035_1arc_v3_cropped_test_keep_n0.05_seed5.pt",map_location="cpu",weights_only=False)
trainset = torch.load(r"./cache/trainset_2_M_points_5_nei_new_saving_with_batching.pt",map_location="cpu",weights_only=False)
validset = torch.load(r"./cache/validset_2_M_points_5_nei_new_saving_with_batching.pt",map_location="cpu",weights_only=False)
testset = torch.load(r"./cache/testset_2_M_points_5_nei_new_saving_with_batching.pt",map_location="cpu",weights_only=False)

/home/user_116/Project-B-Technion/Transformer_Map_Interp/models
/home/user_116/Project-B-Technion
cache exists? True -> /home/user_116/Project-B-Technion/Transformer_Map_Interp/cache


In [5]:
avg_size = sum(len(inner > 0) for inner in validset.obs_coords_norm) / len(validset.obs_coords_norm)
print(f"Number of neighbors at Val set: {avg_size}")
print(f"size of validset: {len(validset.obs_coords_norm)}")
print(avg_size)

avg_size = sum(len(inner > 0) for inner in testset.obs_coords_norm) / len(testset.obs_coords_norm)
print(f"Number of neighbors at Test set: {avg_size}")
print(f"size of testset: {len(testset.obs_coords_norm)}")

avg_size = sum(len(inner) for inner in trainset.obs_coords_norm) / len(trainset.obs_coords_norm)
print(f"Number of neighbors at Train set:{avg_size}")
print(f"size of trainset: {len(trainset.obs_coords_norm)}")

print(trainset.y_std.detach().cpu().numpy())
print(trainset.y_mean.detach().cpu().numpy())

print("trainset obs coords norm:", trainset.obs_coords_norm.shape)
#print("trainset obs coords norm:", trainset.obs_coords_norm[1])

print("trainset obs coords:", trainset.obs_coords.shape)
print("trainset obs coords norm:", trainset.obs_coords[1])

Number of neighbors at Val set: 10.0
size of validset: 81022
10.0
Number of neighbors at Test set: 10.0
size of testset: 81022
Number of neighbors at Train set:10.0
size of trainset: 2431575
300.9501
274.30548
trainset obs coords norm: torch.Size([2431575, 10, 2])
trainset obs coords: torch.Size([2431575, 10, 2])
trainset obs coords norm: tensor([[32.5773, 35.4538],
        [32.5726, 35.4588],
        [32.5720, 35.4585],
        [32.5751, 35.4521],
        [32.5776, 35.4585],
        [32.5759, 35.4618],
        [32.5732, 35.4604],
        [32.5693, 35.4504],
        [32.5757, 35.4496],
        [32.5754, 35.4535]])


In [6]:
print("Performing Local IDW interpolation on validation and test sets...")
val_query_coords = torch.zeros_like(validset.query_coords)
test_query_coords = torch.zeros_like(testset.query_coords)
val_query_coords_normal = validset.query_coords
val_pred = idw_local_all(validset.obs_coords_norm, validset.obs_y_norm, val_query_coords)
test_pred = idw_local_all(testset.obs_coords_norm, testset.obs_y_norm,
                     test_query_coords)

Performing Local IDW interpolation on validation and test sets...


In [8]:
mse_val = np.mean((val_pred - validset.q_y_norm.detach().cpu().numpy())**2)
mae_val = np.mean(np.abs(val_pred - validset.q_y_norm.detach().cpu().numpy()))
mse_test = np.mean((test_pred - testset.q_y_norm.detach().cpu().numpy())**2)
mae_test = np.mean(np.abs(test_pred - testset.q_y_norm.detach().cpu().numpy()))
print(f"[Local IDW] Validation MSE={mse_val:.5f}, MAE={mae_val:.5f}")
print(f"[Local IDW] Test MSE={mse_test:.5f}, MAE={mae_test:.5f}")

[Local IDW] Validation MSE=0.00678, MAE=0.05770
[Local IDW] Test MSE=0.00397, MAE=0.03529


In [35]:
val_pred_unnorm = val_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
test_pred_unnorm = test_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
#val_true_unnorm = validset.query_y.detach().cpu().numpy() * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
#test_true_unnorm = testset.query_y.detach().cpu().numpy() * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
val_true_unnorm = validset.query_y.detach().cpu().numpy()
test_true_unnorm = testset.query_y.detach().cpu().numpy()
print(val_pred_unnorm)
print(test_pred_unnorm)
mse_val = np.mean((val_pred_unnorm -val_true_unnorm)**2)
print(validset.query_y)
mae_val = np.mean(np.abs(val_pred_unnorm - val_true_unnorm))
mse_test = np.mean((test_pred_unnorm -test_true_unnorm)**2)
mae_test = np.mean(np.abs(test_pred_unnorm - test_true_unnorm))
print(f"[Local IDW] Validation MSE={mse_val:.3f}, MAE={mae_val:.3f}")
print(f"[Local IDW] Test MSE={mse_test:.3f}, MAE={mae_test:.3f}")

[-190.16942969  702.9249971  -223.12797545 ... -286.78999814  818.35420448
  656.51344165]
[6.73076408e+02 6.74903667e+02 4.95749883e+02 ... 1.05447491e+02
 1.20122202e+02 5.18678906e-02]
tensor([-187.,  676., -233.,  ..., -288.,  809.,  691.])
[Local IDW] Validation MSE=614.450, MAE=17.366
[Local IDW] Test MSE=359.963, MAE=10.620


In [9]:
print("Performing Local Linear interpolation on validation and test sets...")
val_query_coords = torch.zeros_like(validset.query_coords)
test_query_coords = torch.zeros_like(testset.query_coords)
val_query_coords_normal = validset.query_coords
val_pred = linear_local(validset.obs_coords_norm, validset.obs_y_norm, val_query_coords)
test_pred = linear_local(testset.obs_coords_norm, testset.obs_y_norm,test_query_coords)

Performing Local Linear interpolation on validation and test sets...
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN: 0
Total NaN so far: 1
Total NaN: 1
Total NaN: 1
Total NaN: 1
Total NaN: 1
Total NaN: 1
Total NaN: 1
Total NaN: 1
Total NaN so far: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total NaN: 2
Total N

In [10]:
mse_val = np.mean((val_pred - validset.q_y_norm.detach().cpu().numpy())**2)
mae_val = np.mean(np.abs(val_pred - validset.q_y_norm.detach().cpu().numpy()))
mse_test = np.mean((test_pred - testset.q_y_norm.detach().cpu().numpy())**2)
mae_test = np.mean(np.abs(test_pred - testset.q_y_norm.detach().cpu().numpy()))
print(f"[Local Linear] Validation MSE={mse_val:.5f}, MAE={mae_val:.5f}")
print(f"[Local Linear] Test MSE={mse_test:.5f}, MAE={mae_test:.5f}")

[Local Linear] Validation MSE=0.00578, MAE=0.05202
[Local Linear] Test MSE=0.00318, MAE=0.03055


In [11]:
val_pred_unnorm = val_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
test_pred_unnorm = test_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
val_true_unnorm = validset.query_y.detach().cpu().numpy() * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
test_true_unnorm = testset.query_y.detach().cpu().numpy() * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
print(val_pred_unnorm)
print(test_pred_unnorm)
mse_val = np.mean((val_pred_unnorm -val_true_unnorm)**2)
print(validset.query_y)
mae_val = np.mean(np.abs(val_pred_unnorm - val_true_unnorm))
mse_test = np.mean((test_pred_unnorm -test_true_unnorm)**2)
mae_test = np.mean(np.abs(test_pred_unnorm - test_true_unnorm))
print(f"[Local Linear] Validation MSE={mse_val:.3f}, MAE={mae_val:.3f}")
print(f"[Local Linear] Test MSE={mse_test:.3f}, MAE={mae_test:.3f}")

[ 489.70974461  689.89510262  774.08324371 ...  747.86639631 -289.58180702
 -290.39828826]
[357.62976788 377.09783844 661.88726806 ... 659.0406655  265.50422843
  70.85505058]
tensor([ 0.6131,  1.3823,  1.6608,  ...,  1.4718, -1.8736, -1.8603])
[Local Linear] Validation MSE=526.104, MAE=15.691
[Local Linear] Test MSE=289.339, MAE=9.216


In [12]:
print("Performing Local RBF interpolation on validation and test sets...")
val_query_coords = torch.zeros_like(validset.query_coords)
test_query_coords = torch.zeros_like(testset.query_coords)
val_query_coords_normal = validset.query_coords
val_pred = rbf_local(validset.obs_coords_norm, validset.obs_y_norm, val_query_coords)
test_pred = rbf_local(testset.obs_coords_norm, testset.obs_y_norm,test_query_coords)

Performing Local RBF interpolation on validation and test sets...


In [13]:
mse_val = np.mean((val_pred - validset.q_y_norm.detach().cpu().numpy())**2)
mae_val = np.mean(np.abs(val_pred - validset.q_y_norm.detach().cpu().numpy()))
mse_test = np.mean((test_pred - testset.q_y_norm.detach().cpu().numpy())**2)
mae_test = np.mean(np.abs(test_pred - testset.q_y_norm.detach().cpu().numpy()))
print(f"[Local RBF] Validation MSE={mse_val:.3f}, MAE={mae_val:.3f}")
print(f"[Local RBF] Test MSE={mse_test:.4f}, MAE={mae_test:.4f}")

[Local RBF] Validation MSE=0.011, MAE=0.078
[Local RBF] Test MSE=0.0054, MAE=0.0513


In [14]:
val_pred_unnorm = val_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
test_pred_unnorm = test_pred * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
val_true_unnorm = validset.query_y.detach().cpu().numpy() * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
test_true_unnorm = testset.query_y.detach().cpu().numpy() * trainset.y_std.detach().cpu().numpy() + trainset.y_mean.detach().cpu().numpy()
print(val_pred_unnorm)
print(test_pred_unnorm)
mse_val = np.mean((val_pred_unnorm -val_true_unnorm)**2)
print(validset.query_y)
mae_val = np.mean(np.abs(val_pred_unnorm - val_true_unnorm))
mse_test = np.mean((test_pred_unnorm -test_true_unnorm)**2)
mae_test = np.mean(np.abs(test_pred_unnorm - test_true_unnorm))
print(f"[Local RBF] Validation MSE={mse_val:.3f}, MAE={mae_val:.3f}")
print(f"[Local RBF] Test MSE={mse_test:.3f}, MAE={mae_test:.3f}")

[ 461.63347254  651.47308375  772.74983816 ...  718.84307964 -242.73886124
 -263.84146428]
[356.64709233 361.36517417 679.45576335 ... 623.30450276 264.67719234
  76.19148461]
tensor([ 0.6131,  1.3823,  1.6608,  ...,  1.4718, -1.8736, -1.8603])
[Local RBF] Validation MSE=979.565, MAE=23.561
[Local RBF] Test MSE=486.698, MAE=15.473


In [ ]:
from matplotlib.colors import Normalize, TwoSlopeNorm

# -------------------------------------
# PICK QUERY SAMPLE
# -------------------------------------
i = np.random.randint(len(testset))  # random query from test
q_coord = testset.query_coords[i]    # (lat, lon)
q_gt = testset.query_y[i].item()
obs_coords = testset.obs_coords[i]
obs_y = testset.obs_y[i]
device = 'cuda'
# -------------------------------------
# MODEL PREDICTION
# -------------------------------------
# normalize neighbor coords relative to query
obs_coords_norm = testset.obs_coords_norm[i]
print(len(obs_coords_norm))
obs_y_norm = testset.obs_y_norm[i]

# normalize lat/lon by std
dlat = obs_coords_norm[:, 0] 
dlon = obs_coords_norm[:, 1] 

mem_tokens = torch.stack([dlat, dlon, obs_y_norm.squeeze(-1)], dim=-1).unsqueeze(0).to(device)
print(mem_tokens)
pad_mask = torch.zeros(mem_tokens.size(1), dtype=torch.bool, device=device).unsqueeze(0)

with torch.no_grad():
    y_pred_norm = model(mem_tokens, pad_mask)
y_pred = y_pred_norm * trainset.y_std + trainset.y_mean

print(f"\nQuery #{i}: GT={q_gt:.2f}  |  Pred={y_pred.item():.2f}")

# -------------------------------------
# PLOT MAP + POINTS
# -------------------------------------
elev_min = float(min(trainset.y.min(), testset.y.min()))
elev_max = float(max(trainset.y.max(), testset.y.max()))
elev_norm = Normalize(vmin=elev_min, vmax=elev_max)

plt.figure(figsize=(10, 8))

# === ADD TEST SET POINTS ===
# sc = plt.scatter(trainset.coords[:, 1], trainset.coords[:, 0],
#                  c=trainset.y.squeeze(-1), norm=elev_norm, s=2, cmap="terrain", alpha=0.5)

plt.scatter(testset.coords[:, 1], testset.coords[:, 0],
            c=testset.y.squeeze(-1), norm=elev_norm, s=2, cmap="terrain", alpha=0.5)
plt.colorbar(label="Elevation (m)")
# plt.colorbar(sc, label="Elevation (m)")

# query GT and prediction
plt.scatter(q_coord[1], q_coord[0], c='red', s=5, edgecolors='red', label=f"GT={q_gt:.1f}, Pred={y_pred.item():.1f}")
plt.scatter(obs_coords[:,1], obs_coords[:,0], c='green', s=5, edgecolors='k', label="Neighbors")
# plt.scatter(q_coord[1], q_coord[0], c='red', marker='x', s=100, label=f"Pred={y_pred.item():.1f}")

plt.title(f"Query #{i}: GT vs Pred with Neighbors")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.show()

# ZOOM IN
plt.figure()
lat_c, lon_c = float(q_coord[0]), float(q_coord[1])
plt.xlim(lon_c - 0.1, lon_c +0.1)
plt.ylim(lat_c - 0.1, lat_c + 0.1)
plt.scatter(testset.coords[:, 1], testset.coords[:, 0],
            c=testset.y.squeeze(-1), norm=elev_norm, s=2, cmap="terrain", alpha=0.5)
plt.colorbar(label="Elevation (m)")
# plt.colorbar(sc, label="Elevation (m)")

# query GT and prediction
plt.scatter(q_coord[1], q_coord[0], c='red', s=5, edgecolors='red', label=f"GT={q_gt:.1f}, Pred={y_pred.item():.1f}")
plt.scatter(obs_coords[:,1], obs_coords[:,0], c='green', s=5, edgecolors='k', label="Neighbors")
# plt.scatter(q_coord[1], q_coord[0], c='red', marker='x', s=100, label=f"Pred={y_pred.item():.1f}")

plt.title(f"Query #{i}: GT vs Pred with Neighbors")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()

plt.gca().set_aspect('equal', adjustable='box')
plt.show()
